短期记忆记录的是会话级别（线程，Thread） 的数据，会话间不共享。

而长期记忆记录的是用户特定或应用级别的数据，任何会话都可以随时访问

#  类型划分

LangChain参考[CoALA paper](https://arxiv.org/pdf/2309.02427)将长期记忆划分为三类：

| Memory Type | 存什么 |
| --- | --- |
| [Semantic](https://docs.langchain.com/oss/python/concepts/memory#semantic-memory)（语义记忆） | 事实 |
| [Episodic](https://docs.langchain.com/oss/python/concepts/memory#episodic-memory)（情景记忆） | 经验 |
| [Procedural](https://docs.langchain.com/oss/python/concepts/memory#procedural-memory)（程序性记忆） | 规则/做事方法 |

## 类型1：Semantic Memory（语义记忆）

即“事实类记忆”，记录事实/用户偏好/概念，如：

- 用户喜欢简洁回答
- 用户常用中文
- 某个公司属于哪个行业

## 类型2：Episodic Memory（情景记忆）

即“经验类记忆”，记录Agent过去执行的动作，如：

- 过去某个任务是怎么成功的
- 某种用户输入下，怎样回答效果最好

在Agent里，这常常表现为 few‑shot examples（少样本示例）：
不直接告诉模型规则，而是给它看几个“输入 -> 输出”的例子，让它照着学

## 类型3：Procedural Memory（程序性记忆）

即“规则/做事方法”，如

- Agent的系统提示词
- Agent的工作流程
- 工具调用规则

# 3.1.3 存储架构

长期记忆的存储是 `store -> namespace -> key -> value` 的四层架构。

### 第1层：Store（记忆仓库）

- Store是 `langgraph.store.base.BaseStore` 的子类实例，由全类名可知，store是由LangGraph提供的。
  - 常用实现类：
    - `InMemoryStore`：将长期记忆存储在内存，适合测试
    - `PostgresStore`：将长期记忆存储在外部的PostgreSQL数据库，适合生产环境
- 开发期可用 InMemoryStore；生产建议数据库后端，如 PostgresStore

### 第2层：Namespace（命名空间）

数据类型是由任意长度的 `tuple[str, ...]` 表示的**层级路径**。作用上很像“文件路径 / 文件夹层级”，用于给长期记忆分组和隔离。数据类型为**字符串元组**

### 第3层：Key（键）

是该 namespace 下的唯一标识，单条记忆的唯一键，数据类型为**字符串(str)**

### 第4层：Value（值）

是存储的值，数据类型为**字典(dict[str, Any])**

## 举例1：

每个namespace存储的都是key‑value键值对，通过key可以唯一标识一条value。

```
namespace = ("users", "user_123", "preferences") # 元组类型
key = "profile"        # 字符串类型
value = {               # 字典类型
    "language": "zh‑CN",
    "style": "short_direct",
    "likes": ["python", "rag"]
}
store.put(namespace, key, value)
```

## 举例2：

同一个 AI 应用通常会为每个独立会话维护各自的短期状态 State；而长期记忆通常可以共享同一个 Store 实例，再通过 namespace 区分不同用户、组织、业务域或会话相关数据。

```
AI应用
├─ thread_id = t1 -> state_1
│  ├─ messages = [
│  │    {"role": "user", "content": "我想去北京旅游"},
│  │    {"role": "assistant", "content": "你想玩几天？"}
│  │ ]
│  ├─ current_intent = "travel_planning"
│  └─ collected_slots = {"destination": "北京"}
│
├─ thread_id = t2 -> state_2
│  ├─ messages = [
│  │    {"role": "user", "content": "帮我写周报"}
│  │ ]
│  ├─ current_intent = "write_report"
│  └─ collected_slots = {}
│
├─ thread_id = t3 -> state_3
│  ├─ messages = [
│  │    {"role": "user", "content": "我喜欢简洁风格的UI"}
│  │ ]
│  ├─ current_intent = "ui_design"
│  └─ collected_slots = {"style": "minimal"}
│
└─ shared store
   ├─ namespace = (user_1, "memories")
   │  ├─ key = "profile"
   │  │  value = {
   │  │     "name": "张三",
   │  │     "city": "上海",
   │  │     "preferences": ["简洁风格", "中文回复"]
   │  │  }
   │  ├─ key = "travel_preference"
   │  │  value = {
   │  │     "favorite_cities": ["北京", "杭州"],
   │  │     "budget_level": "medium"
   │  │  }
   │  └─ key = "writing_style"
   │     value = {
   │        "language": "zh‑CN",
   │        "tone": "formal"
   │     }
   │
   ├─ namespace = (user_2, "memories")
   │  ├─ key = "profile"
   │  │  value = {
   │  │     "name": "李四",
   │  │     "city": "深圳"
   │  │  }
   │  └─ key = "product_interest"
   │     value = {
   │        "topics": ["AI Agent", "RAG", "workflow"]
   │     }
   │
   └─ namespace = (user_1, thread_3, "artifacts")
      ├─ key = "draft_v1"
      │  value = {
      │     "type": "html",
      │     "content": "<html>...</html>"
      │  }
      ├─ key = "ui_notes"
      │  value = {
      │     "summary": "用户偏好留白多、低饱和配色"
      │  }
      └─ key = "final_scheme"
         value = {
            "palette": ["#F5E8E8", "#E3A5E5", "#D976D2"],
            "font_style": "clean"
         }
```